# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aanvi02/ML-work/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Can historical content performance data help identify content opportunities that should be prioritized for further review?

### Decision Supported

The analysis will help the content team rank potential opportunities for review and decide which ones deserve further investigation, using only information available at the time of the decision.

### Objective

The objective is to evaluate whether a simple, transparent baseline and a machine-learning model can support this prioritization task. The approach will be selected after checking the available data, defining a valid target, and testing for data leakage.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Dataset and Release

- **Source:** FlyRank Internship Warehouse
- **Release/version:** [Fill in the actual release used]
- **Tables used:** [List the tables actually queried]
- **Date window:** [Enter the verified start and end dates]
- **Rows analysed:** [Enter the actual row count]

### Exclusions and Rationale

- Exclude records with missing or invalid values in fields required for the analysis, where appropriate.
- Exclude features that would reveal future information or leak the target into model training.
- Exclude personally identifying, client-identifying, or otherwise restricted information from public outputs.
- Document any additional exclusions only after inspecting the actual data.

### Privacy and Public Safety

Only report aggregated, non-identifying results. Do not publish raw restricted records, sensitive queries, or client-identifying details.

In [1]:
from pathlib import Path
import subprocess
import os

REPO = Path("/content/ML-work")

if not (REPO / "scripts" / "run_all.py").exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Aanvi02/ML-work.git",
            str(REPO)
        ],
        check=True
    )

os.chdir(REPO)

print("Repository ready:", REPO)
print("Pipeline exists:",
      (REPO / "scripts" / "run_all.py").exists())

Repository ready: /content/ML-work
Pipeline exists: True


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology

**Assumptions**

The analysis assumes that the available historical data is sufficiently reliable for evaluating the selected research question. These assumptions will be checked during data exploration.

**Feature Selection**

Features will be selected based on their relevance to the business decision and whether they would have been available at the time the prediction is made. Identifiers and features containing future information will not be used as predictive inputs.

**Label Definition**

The target label will be defined using an observed outcome in the dataset. Its meaning, time window, and availability will be verified before model training. No target label will be invented.

**Baseline**

A simple rule-based or majority-class baseline will be established before comparing it with a machine-learning model.

**Validation Design**

Training and evaluation data will be separated to estimate performance on unseen observations. Where records have a time component, a time-aware split will be considered to reflect the intended use case.

**Leakage Checks**

The analysis will check for target leakage, future information in features, duplicate records across splits, and inappropriate use of identifiers. Any detected issues will be documented and addressed before final evaluation.

In [2]:
from pathlib import Path

REPO = Path("/content/ML-work")

print("Repository files:")
print([p.name for p in REPO.iterdir()])

print("\nScripts folder:")
scripts = REPO / "scripts"
print([p.name for p in scripts.iterdir()] if scripts.exists() else "Not found")

print("\nRaw data folder:")
raw = REPO / "data" / "raw"
print([p.name for p in raw.iterdir()] if raw.exists() else "Not found")

Repository files:
['submission', 'AGENTS.md', '.gitignore', '.git', 'SETUP.md', 'requirements.txt', 'notebooks', 'DATA_USE.md', '.github', 'work', 'CLAUDE.md', 'skills', 'GUIDE.md', 'outputs', 'README.md', 'data', 'scripts', 'docs', 'LICENSE']

Scripts folder:
['run_all.py', '02_baseline_score.py', 'ml_utils.py', '03_train_model.py', '05_build_pdf_report.py', '04_evaluate_and_export.py', '01_prepare_features.py']

Raw data folder:
['content_refresh_anonymized.csv']


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Results

The pipeline evaluated three machine-learning models using a client-holdout split. The dataset contained 30,000 rows, with 27,675 rows used for training and 2,325 for testing.

**Model comparison**

| Model | Accuracy | F1-score | ROC-AUC | Precision@50 |
|---|---:|---:|---:|---:|
| Baseline | 0.609 | 0.274 | 0.627 | 0.240 |
| Logistic Regression | 0.661 | 0.566 | 0.700 | 0.400 |
| Decision Tree | 0.677 | 0.634 | 0.742 | 0.580 |
| Random Forest | 0.672 | 0.640 | 0.750 | 0.740 |

Random Forest was selected using Precision@50. It achieved 0.74 Precision@50, meaning 37 of the top 50 ranked records in the evaluated test set were labelled as declining. This was higher than the baseline value of 0.24.

The Random Forest also achieved a ROC-AUC of approximately 0.750 and an F1-score of approximately 0.640. The Decision Tree had slightly higher accuracy, at approximately 0.677, while Random Forest performed better on the selected ranking metric.

These results indicate that the model may help prioritize records for human review. However, the target is derived from the existing `trend_direction` field, so this experiment evaluates classification of the recorded trend label rather than prediction of future decline.

In [3]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/ML-work/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values (top 15 columns):")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nPossible target columns:")
print([
    col for col in df.columns
    if any(word in col.lower() for word in ["label", "declin", "trend"])
])

Dataset loaded successfully!
Number of rows: 30000
Number of columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Missing values (top 15 columns):
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699

## 5. Limitations

*What this work cannot claim.*

### Limitations

This analysis has several limitations. First, the findings depend on the quality, completeness, and representativeness of the available historical dataset. Missing values, inconsistent records, or limited coverage may affect the reliability of the results.

Second, historical patterns may not reflect future content performance. Changes in audience behaviour, content strategy, and external factors may reduce the usefulness of patterns learned from past data.

Third, the model's performance depends on the definition and availability of the target label and on whether the selected features would be available at prediction time. Potential data leakage must be checked before interpreting evaluation results.

Finally, model outputs should be treated as decision-support signals rather than guaranteed predictions. Recommendations require further review, and conclusions should be limited to what the data and evaluation results actually support.

In [5]:


import pandas as pd
from pathlib import Path

DATA = Path("/content/ML-work/data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA)

print("Dataset shape:", df.shape)

print("\nAll columns:")
for col in df.columns:
    print(col)

print("\nTarget label check:")
for col in ["is_declining_label", "trend_direction", "trend_pct"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).head(10))

print("\nDataset preview (first 5 rows):")
display(df.head())

Dataset shape: (30000, 44)

All columns:
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct

Target label check:

trend_direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct:
trend_pct
 NaN      3388
-100.0    1395
 0.0       443
-50.0      281
-66.7      172
 100.0     145
-33.3      137
-75.0       98
-80.0       96
 50

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Ranked Recommendations

The following recommendations will be finalized after evaluating the model, comparing it with the baseline, and examining the supporting evidence.

**Priority 1 — Validate the highest-ranked opportunities**

Review the highest-ranked items against the available historical evidence before taking action. Confirm that the ranking is reliable and that the selected items meet the team's practical requirements.

**Priority 2 — Compare recommendations with the baseline**

Determine whether the model produces a more useful ranking than the baseline. If it does not demonstrate an improvement, retain the simpler approach.

**Priority 3 — Investigate the factors associated with outcomes**

Examine relevant features and patterns to understand why certain items receive higher scores. Treat these patterns as associations rather than proof of causation.

**Priority 4 — Monitor performance over time**

Re-evaluate the approach on later data and monitor whether the recommendations remain useful as conditions change.

**Priority 5 — Maintain privacy and human review**

Use aggregated, non-identifying evidence in public reporting. Keep final business decisions subject to human review rather than relying on model scores alone.

### Evidence and Prioritization

Each recommendation will be linked to a verified result, a measured limitation, or an explicitly stated operational requirement. No ranking or performance improvement will be claimed without supporting evidence.

In [6]:
# Step 5: Data quality checks

print("1. Dataset size:", df.shape)

print("\n2. Duplicate rows:", df.duplicated().sum())

print("\n3. Total missing values:", df.isnull().sum().sum())

print("\n4. Columns with most missing values:")
print(
    df.isnull().sum()
      .sort_values(ascending=False)
      .head(10)
)

target = "is_declining_label"

if target in df.columns:
    print("\n5. Target label distribution:")
    print(df[target].value_counts(dropna=False))

    print("\nTarget label percentages:")
    print(
        (df[target].value_counts(normalize=True, dropna=False) * 100)
        .round(2)
    )
else:
    print("\nTarget label not found:", target)
    print("Review the available columns before model training.")

1. Dataset size: (30000, 44)

2. Duplicate rows: 0

3. Total missing values: 73868

4. Columns with most missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
dtype: int64

Target label not found: is_declining_label
Review the available columns before model training.


### 7. Artifacts

The pipeline generated the following artifacts:

- **Prepared dataset:** `data/processed/refresh_feature_vector.csv`
- **Baseline ranked queue:** `data/processed/baseline_refresh_queue.csv`
- **Model predictions:** `data/processed/model_predictions.csv`
- **Evaluation results:** `outputs/model_results.json`
- **Ranked refresh queue:** `outputs/refresh_queue.csv`
- **Model evaluation report:** `outputs/model_report.md`
- **Evaluation charts:** `outputs/charts/`
- **PDF report:** `outputs/flyrank_refresh_model_results.pdf`

These artifacts support reproducibility, model comparison, review of ranked opportunities, and communication of the findings.

### 6. Ranked Recommendations

Based on the evaluation results, the following actions are recommended:

1. **Prioritize the top-ranked records for human review.** The Random Forest achieved a Precision@50 of 0.74 on the held-out test set. Review these records first to identify potential content refresh opportunities.

2. **Review content performance and visibility.** The most influential model features included `days_with_impressions`, `log_impressions_90d`, `avg_position`, and `content_age_days`. Review these signals alongside the content's actual performance before deciding whether an update is needed.

3. **Use the model as a decision-support tool.** A high score should trigger further investigation, not an automatic content update. Human reviewers should consider relevance, quality, search intent, and business context.

4. **Validate before wider use.** Evaluate the approach on additional held-out data and monitor whether ranking quality remains stable over time.

5. **Avoid claiming future prediction.** The current label is derived from `trend_direction == "down"`. The model classifies the recorded trend label; it has not yet demonstrated that it can predict future content decline.

These recommendations are based on the available evaluation results and should be treated as a starting point for further review.

In [7]:


print("Columns related to trends, labels, or outcomes:")

candidates = [
    col for col in df.columns
    if any(word in col.lower()
           for word in ["trend", "declin", "label", "outcome", "target"])
]

print(candidates)

for col in candidates:
    print(f"\n--- {col} ---")
    print("Data type:", df[col].dtype)
    print("Unique values:", df[col].nunique(dropna=True))
    print(df[col].value_counts(dropna=False).head(10))

Columns related to trends, labels, or outcomes:
['trend_direction', 'trend_pct']

--- trend_direction ---
Data type: object
Unique values: 5
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

--- trend_pct ---
Data type: float64
Unique values: 2712
trend_pct
 NaN      3388
-100.0    1395
 0.0       443
-50.0      281
-66.7      172
 100.0     145
-33.3      137
-75.0       98
-80.0       96
 50.0       88
Name: count, dtype: int64


In [8]:


df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

print("Target created successfully!")
print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget percentages:")
print(
    (df["is_declining_label"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nLeakage columns to exclude from model features:")
print(["trend_direction", "trend_pct"])

Target created successfully!

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64

Leakage columns to exclude from model features:
['trend_direction', 'trend_pct']


In [9]:
# Step 8: Inspect the provided project pipeline

from pathlib import Path

pipeline_file = Path("/content/ML-work/scripts/run_all.py")

print("Pipeline file exists:", pipeline_file.exists())

if pipeline_file.exists():
    print("\nPipeline instructions:\n")
    print(pipeline_file.read_text()[:10000])

Pipeline file exists: True

Pipeline instructions:

from __future__ import annotations

import subprocess
import sys
from pathlib import Path

from ml_utils import OUTPUT_DIR, RAW_PATH, ROOT, read_json


STEPS = [
    ("01_prepare_features.py", "Prepare features — clean the data, build the feature vector, define the label"),
    ("02_baseline_score.py", "Baseline — a transparent hand-written rule to beat"),
    ("03_train_model.py", "Train — logistic regression, decision tree, random forest (client-holdout split)"),
    ("04_evaluate_and_export.py", "Evaluate — ranked refresh queue, charts, and the Markdown report"),
    ("05_build_pdf_report.py", "Report — a shareable PDF summary"),
]


def run_step(index: int, script: str, label: str) -> None:
    print(f"\n{'=' * 70}\n▶ Step {index}/{len(STEPS)} — {label}\n{'=' * 70}", flush=True)
    subprocess.run([sys.executable, str(ROOT / "scripts" / script)], cwd=ROOT, check=True)


def main() -> None:
    if not RAW_PATH.exists():
        rai

In [10]:
# Step 9: Check the project's required libraries

from pathlib import Path

REPO = Path("/content/ML-work")

print("Requirements file:")
print((REPO / "requirements.txt").read_text())

print("\nFeature preparation script:")
print((REPO / "scripts" / "01_prepare_features.py").read_text()[:8000])

Requirements file:
pandas>=2.2
numpy>=1.26
scikit-learn>=1.4
matplotlib>=3.8
reportlab>=4.0
duckdb>=1.0
huggingface_hub>=0.24


Feature preparation script:
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_utils import (
    BOOLEAN_COLUMNS,
    CATEGORICAL_COLUMNS,
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_COLUMNS,
    PROCESSED_DIR,
    RAW_PATH,
    display_path,
    ensure_dirs,
    to_bool_series,
    write_json,
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Prepare FlyRank refresh feature vector.")
    parser.add_argument("--input", default=str(RAW_PATH), help="Raw anonymized CSV export.")
    parser.add_argument(
        "--output",
        default=str(PROCESSED_DIR / "refresh_feature_vector.csv"),
        help="Prepared feature-vector CSV.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
   

In [12]:
import sys
sys.path.insert(0, "/content/ML-work/scripts")

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

print("Numeric:", MODEL_NUMERIC_FEATURES)
print("Categorical:", MODEL_CATEGORICAL_FEATURES)

Numeric: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


In [13]:
%cd /content/ML-work
!python scripts/run_all.py

/content

▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/ML-work/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/ML-work/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/ML-work/data/processed/model_predictions.csv
Wrote model results: /content/ML-work/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/ML-work/outputs/refresh_queue.csv
Wrote model report: /content/ML-work/outputs/model_report.md
Wrote charts in: /content/ML-work

In [14]:
%pip install reportlab -q

In [15]:
%cd /content/ML-work
!python scripts/05_build_pdf_report.py

/content
Wrote PDF report: /content/ML-work/outputs/flyrank_refresh_model_results.pdf


In [16]:
import json
from pathlib import Path

p = Path("/content/ML-work/outputs/model_results.json")
results = json.loads(p.read_text())

print(json.dumps(results, indent=2))

{
  "baseline": {
    "baseline_accuracy": 0.6086021505376344,
    "baseline_average_precision": 0.46760658636934826,
    "baseline_f1": 0.2743221690590112,
    "baseline_precision": 0.4985507246376812,
    "baseline_precision_at_100": 0.36,
    "baseline_precision_at_20": 0.15,
    "baseline_precision_at_50": 0.24,
    "baseline_recall": 0.18921892189218922,
    "baseline_roc_auc": 0.6268917852237201
  },
  "best_model": {
    "feature_importance_top": [
      {
        "feature": "days_with_impressions",
        "importance": 0.15814381552310994
      },
      {
        "feature": "log_impressions_90d",
        "importance": 0.12863800983285106
      },
      {
        "feature": "avg_position",
        "importance": 0.10916404306325882
      },
      {
        "feature": "content_age_days",
        "importance": 0.0951682857931063
      },
      {
        "feature": "char_count",
        "importance": 0.042608309447693275
      },
      {
        "feature": "word_count",
        "im

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
